In [0]:
data = [
(1, "Bitcoin", "2024-02-01", 430, 435),
(1, "Bitcoin", "2024-02-02", 435, 440),
(1, "Bitcoin", "2024-02-03", 440, 439),
(1, "Bitcoin", "2024-02-04", 445, 430),
(2, "Ethereum", "2024-02-01", 320, 325),
(2, "Ethereum", "2024-02-02", 325, 330),
(2, "Ethereum", "2024-02-03", 330, 335),
(2, "Ethereum", "2024-02-04", 335, 340),
(3, "Solana", "2024-02-01", 120, 125),
(3, "Solana", "2024-02-02", 125, 130),
(3, "Solana", "2024-02-03", 130, 135),
(3, "Solana", "2024-02-04", 135, 140)
]
columns = ["Coin_ID", "Coin_Name", "Date", "Open_Price", "Close_Price"]

df = spark.createDataFrame(data,columns)

In [0]:
df.display()

Coin_ID,Coin_Name,Date,Open_Price,Close_Price
1,Bitcoin,2024-02-01,430,435
1,Bitcoin,2024-02-02,435,440
1,Bitcoin,2024-02-03,440,439
1,Bitcoin,2024-02-04,445,430
2,Ethereum,2024-02-01,320,325
2,Ethereum,2024-02-02,325,330
2,Ethereum,2024-02-03,330,335
2,Ethereum,2024-02-04,335,340
3,Solana,2024-02-01,120,125
3,Solana,2024-02-02,125,130


In [0]:
from pyspark.sql import functions as F

In [0]:
df1 = df.withColumn("Date",F.to_date(F.col("Date"),"yyyy-MM-dd"))
df1.display()

Coin_ID,Coin_Name,Date,Open_Price,Close_Price
1,Bitcoin,2024-02-01,430,435
1,Bitcoin,2024-02-02,435,440
1,Bitcoin,2024-02-03,440,439
1,Bitcoin,2024-02-04,445,430
2,Ethereum,2024-02-01,320,325
2,Ethereum,2024-02-02,325,330
2,Ethereum,2024-02-03,330,335
2,Ethereum,2024-02-04,335,340
3,Solana,2024-02-01,120,125
3,Solana,2024-02-02,125,130


In [0]:
from pyspark.sql.window import Window

In [0]:
window_part = Window.partitionBy("Coin_Name").orderBy("Date")
df2 = (
    df1.withColumn("diff1",F.lag("CLose_Price",1).over(window_part))
        .withColumn("diff2",F.lag("Close_price",2).over(window_part))
)
df2.display()

Coin_ID,Coin_Name,Date,Open_Price,Close_Price,diff1,diff2
1,Bitcoin,2024-02-01,430,435,null,null
1,Bitcoin,2024-02-02,435,440,435,null
1,Bitcoin,2024-02-03,440,439,440,435
1,Bitcoin,2024-02-04,445,430,439,440
2,Ethereum,2024-02-01,320,325,null,null
2,Ethereum,2024-02-02,325,330,325,null
2,Ethereum,2024-02-03,330,335,330,325
2,Ethereum,2024-02-04,335,340,335,330
3,Solana,2024-02-01,120,125,null,null
3,Solana,2024-02-02,125,130,125,null


In [0]:
df3 = df2.filter((F.col("Close_Price") > F.col("diff1")) & (F.col("diff1")>F.col("diff2"))).select("Coin_ID","Coin_Name").distinct()
df3.display()

Coin_ID,Coin_Name
2,Ethereum
3,Solana
